In [0]:
%sql
create Schema if not exists workspace.olist_gold

In [0]:
df_orders = spark.read.table("workspace.olist_silver.orders_customer")
df_payments = spark.read.table("workspace.olist_silver.orders_payment_totals")

display(df_orders.count(),'',df_payments.count())



In [0]:
paymante_validate_df = df_orders.join(df_payments, on = 'order_id', how = 'left')
display(paymante_validate_df.count())
#

In [0]:
%sql
select * from workspace.olist_silver.orders_payment_totals
where order_id in ( "809a282bbd5dbcabb6f2f724fca862ec","
bfbd0f9bdef84302105ad712db648a6c")

In [0]:
%sql
select order_id, count(*) 
from workspace.olist_silver.orders_customer
group by order_id
having count(*) > 1 

In [0]:
paymante_validate_df.printSchema()

In [0]:
fact_order = paymante_validate_df.select("order_id","order_status","customer_unique_id", "order_purchase_timestamp","total_payment")
fact_order = fact_order.withColumnRenamed("order_purchase_timestamp","order_date").withColumnRenamed("customer_unique_id","customer_id")
fact_order.display()

fact_order.write.mode("overwrite").saveAsTable("workspace.olist_gold.fact_order")

In [0]:
%sql
select customer_id, count(*)
 from workspace.olist_gold.fact_order
 group by customer_id
 having count(*) > 1

In [0]:

from pyspark.sql import functions as F

average_order_value = (
    fact_order
    .groupBy("customer_id")
    .agg(
        F.sum("total_payment").alias("total_spend"),
        F.count("total_payment").alias("total_orders")
    )
    .withColumn("avg_order_value", 
                    F.col("total_spend") / F.col("total_orders")
                )
)
display(average_order_value)





In [0]:
average_order_value.count()

In [0]:
%sql
select count(distinct customer_id) from workspace.olist_gold.fact_order


In [0]:
average_order_value.write.mode("overwrite").saveAsTable("workspace.olist_gold.customer_order_metrics")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.window import Window

customers = spark.read.table("workspace.olist_silver.customers")
customer_orders = spark.read.table("workspace.olist_silver.orders_customer")

dim_customer = customer_orders.alias("co").select(col("co.order_id"),col("co.customer_id"), col("co.order_purchase_timestamp")).join(customers.alias("c"), on = "customer_id", how = "left")


dim_customer_partition = Window.partitionBy("customer_unique_id").orderBy(
    F.col("order_purchase_timestamp").desc_nulls_last(),
    F.col("order_id").desc_nulls_last()
)

dim_customer = (dim_customer.withColumn(
                                "rank",
                                F.row_number().over(dim_customer_partition)
                            )
                            .filter(F.col("rank") == 1)
                            .select(col("customer_unique_id").alias("customer_id"),
                                    "customer_city",
                                    "customer_state",
                                    "customer_zip_code_prefix"         
                            )
                )
dim_customer.write.mode("overwrite").saveAsTable("workspace.olist_gold.dim_customer")


In [0]:
%sql
describe detail workspace.olist_gold.dim_customer
